In [ ]:
"""
Script de pseudonymisation CNPS avec Polars
============================================

Conversion du notebook d'anonymisation pour utiliser Polars au lieu de Pandas.
Polars offre des performances supérieures pour les grandes bases de données.

Étapes:
1. Concaténation des fichiers mensuels
2. Extraction des NUMERO_CNPS uniques
3. Génération des ID_ANSTAT (HMAC-SHA256)
4. Jointure et re-split par mois
"""

import polars as pl
import glob
import os
import logging
import secrets
import hashlib
import hmac
from pathlib import Path
from typing import List, Optional, Dict
from datetime import datetime

# ============================================================================
# CONFIGURATION ET LOGGING
# ============================================================================

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

# Configuration des chemins - À MODIFIER ICI UNIQUEMENT
DOSSIER_PARENT = "C:/Users/f.migone/Documents/12-04-2025-19-31-21_files_list"

# Dossiers dérivés automatiquement (ne pas modifier)
DOSSIER_MENSUELS = os.path.join(DOSSIER_PARENT, "source")
DOSSIER_SORTIE = os.path.join(DOSSIER_PARENT, "output")
DOSSIER_MENSUELS_ID = os.path.join(DOSSIER_PARENT, "mensuels")

# ============================================================================
# ÉTAPE 1: CONCATÉNATION DES FICHIERS MENSUELS
# ============================================================================

def lister_fichiers_mensuels(dossier: str = DOSSIER_MENSUELS) -> List[str]:
    """Liste tous les fichiers mensuels CNPS"""
    logger.info(f"📂 Recherche des fichiers dans : {dossier}")
    
    if not Path(dossier).exists():
        logger.error(f"❌ Le dossier {dossier} n'existe pas")
        return []
    
    pattern = os.path.join(dossier, "TRAVAILLEURS_*_*_NUMERO_CNPS.xlsx")
    fichiers = sorted(glob.glob(pattern))
    
    logger.info(f"✓ {len(fichiers)} fichiers mensuels trouvés")
    
    if fichiers:
        logger.info("Liste des fichiers :")
        for f in fichiers:
            nom = Path(f).name
            taille_mb = Path(f).stat().st_size / (1024 * 1024)
            logger.info(f"  - {nom:50s} ({taille_mb:6.2f} MB)")
    
    return fichiers


def charger_excel_polars(chemin_fichier: str) -> pl.DataFrame:
    """Charge un fichier Excel avec Polars"""
    try:
        logger.info(f"📥 Chargement : {Path(chemin_fichier).name}")
        
        # Polars utilise xlsx2csv en arrière-plan pour Excel
        df = pl.read_excel(chemin_fichier)
        
        logger.info(f"   ✓ {df.height:,} lignes × {df.width} colonnes")
        return df
        
    except Exception as e:
        logger.error(f"❌ Erreur lors du chargement de {chemin_fichier}: {e}")
        raise


def concatener_fichiers_mensuels(
    fichiers: List[str],
    nb_fichiers_max: Optional[int] = None
) -> pl.DataFrame:
    """
    Concatène les fichiers mensuels avec Polars
    
    Args:
        fichiers: Liste des chemins de fichiers
        nb_fichiers_max: Nombre maximum de fichiers à traiter
        
    Returns:
        DataFrame Polars concaténé
    """
    logger.info("\n🔗 CONCATÉNATION DES FICHIERS")
    logger.info("-" * 80)
    
    if nb_fichiers_max:
        fichiers = fichiers[:nb_fichiers_max]
        logger.info(f"📌 Seuls les {len(fichiers)} premiers fichiers seront concaténés")
    
    df_list = []
    for i, fichier in enumerate(fichiers, 1):
        logger.info(f"[{i}/{len(fichiers)}] Chargement en cours...")
        df = charger_excel_polars(fichier)
        df_list.append(df)
    
    logger.info("\n   Fusion des DataFrames...")
    df_concat = pl.concat(df_list, how="vertical")
    logger.info(f"   ✓ Base concaténée : {df_concat.height:,} lignes × {df_concat.width} colonnes")
    
    return df_concat


def sauvegarder_polars(
    df: pl.DataFrame,
    nom_fichier: str,
    dossier: str = DOSSIER_SORTIE
) -> Dict[str, str]:
    """
    Sauvegarde un DataFrame Polars en plusieurs formats
    
    Args:
        df: DataFrame Polars
        nom_fichier: Nom de base du fichier
        dossier: Dossier de destination
        
    Returns:
        Dict des chemins créés
    """
    Path(dossier).mkdir(parents=True, exist_ok=True)
    fichiers = {}
    
    nom_base = nom_fichier.replace('.xlsx', '').replace('.parquet', '').replace('.csv', '')
    
    # Parquet (format recommandé pour Polars)
    chemin_parquet = os.path.join(dossier, f"{nom_base}.parquet")
    logger.info(f"💾 Sauvegarde PARQUET : {chemin_parquet}")
    df.write_parquet(chemin_parquet, compression='snappy')
    taille = Path(chemin_parquet).stat().st_size / (1024 * 1024)
    logger.info(f"   ✅ Parquet sauvegardé : {taille:.2f} MB")
    fichiers['parquet'] = chemin_parquet
    
    # CSV
    chemin_csv = os.path.join(dossier, f"{nom_base}.csv")
    logger.info(f"💾 Sauvegarde CSV : {chemin_csv}")
    df.write_csv(chemin_csv)
    taille = Path(chemin_csv).stat().st_size / (1024 * 1024)
    logger.info(f"   ✅ CSV sauvegardé : {taille:.2f} MB")
    fichiers['csv'] = chemin_csv
    
    # Excel (si < 1M lignes)
    if df.height <= 1_048_576:
        chemin_excel = os.path.join(dossier, f"{nom_base}.xlsx")
        logger.info(f"💾 Sauvegarde EXCEL : {chemin_excel}")
        df.write_excel(chemin_excel)
        taille = Path(chemin_excel).stat().st_size / (1024 * 1024)
        logger.info(f"   ✅ Excel sauvegardé : {taille:.2f} MB")
        fichiers['xlsx'] = chemin_excel
    else:
        logger.warning(f"⚠️ {df.height:,} lignes > limite Excel (1,048,576)")
        logger.info("   → Seuls Parquet et CSV seront créés")
    
    return fichiers


# ============================================================================
# ÉTAPE 2: EXTRACTION DES NUMERO_CNPS UNIQUES
# ============================================================================

def extraire_numero_cnps_uniques(
    df: pl.DataFrame,
    col_numero: str = "NUMERO_CNPS"
) -> pl.DataFrame:
    """
    Extrait les NUMERO_CNPS uniques avec Polars
    
    Args:
        df: DataFrame source
        col_numero: Nom de la colonne NUMERO_CNPS
        
    Returns:
        DataFrame avec NUMERO_CNPS uniques
    """
    logger.info("🔍 Extraction des NUMERO_CNPS uniques...")
    
    total = df.height
    logger.info(f"   Total de lignes dans la source : {total:,}")
    
    # Extraction des uniques (très rapide avec Polars)
    df_uniques = (
        df
        .select(pl.col(col_numero).cast(pl.Utf8))
        .unique()
        .sort(col_numero)
    )
    
    uniques = df_uniques.height
    
    logger.info("\n📊 Statistiques :")
    logger.info(f"   Lignes totales              : {total:,}")
    logger.info(f"   NUMERO_CNPS uniques         : {uniques:,}")
    logger.info(f"   Duplications supprimées     : {total - uniques:,}")
    logger.info(f"   Taux de duplication         : {100 * (1 - uniques/total):.2f}%")
    
    # Analyse de structure
    logger.info("\n🔬 Analyse de la structure :")
    
    # Distribution des longueurs
    longueurs = (
        df_uniques
        .select(pl.col(col_numero).str.len_chars().alias("longueur"))
        .group_by("longueur")
        .count()
        .sort("longueur")
    )
    
    logger.info("   Longueurs des NUMERO_CNPS :")
    for row in longueurs.iter_rows(named=True):
        pct = 100 * row['count'] / uniques
        logger.info(f"      {row['longueur']} caractères : {row['count']:,} numéros ({pct:.2f}%)")
    
    # Distribution par sexe (1er caractère)
    distribution_sexe = (
        df_uniques
        .select(pl.col(col_numero).str.slice(0, 1).alias("sexe_code"))
        .group_by("sexe_code")
        .count()
        .sort("sexe_code")
    )
    
    logger.info("\n   Distribution par sexe (1er chiffre) :")
    for row in distribution_sexe.iter_rows(named=True):
        sexe_label = "Homme" if row['sexe_code'] == "1" else "Femme" if row['sexe_code'] == "2" else "Inconnu"
        pct = 100 * row['count'] / uniques
        logger.info(f"      {row['sexe_code']} ({sexe_label}) : {row['count']:,} numéros ({pct:.2f}%)")
    
    logger.info(f"\n   ✓ Extraction terminée : {uniques:,} NUMERO_CNPS uniques")
    
    return df_uniques


# ============================================================================
# ÉTAPE 3: GÉNÉRATION DES ID_ANSTAT
# ============================================================================

def generer_sel_unique() -> str:
    """Génère un sel cryptographique de 32 octets"""
    return secrets.token_hex(32)


def charger_cle_secrete(
    fichier_cle: str = "./cle_secrete_anstat.txt",
    generer_si_absent: bool = True
) -> str:
    """
    Charge ou génère une clé secrète
    
    Args:
        fichier_cle: Chemin du fichier de clé
        generer_si_absent: Générer une clé si absente
        
    Returns:
        Clé secrète
    """
    # Essayer de charger depuis fichier
    if Path(fichier_cle).exists():
        with open(fichier_cle, 'r') as f:
            cle = f.read().strip()
        logger.info(f"✓ Clé secrète chargée depuis : {fichier_cle}")
        return cle
    
    # Essayer variable d'environnement
    cle = os.getenv("ANSTAT_SECRET_KEY")
    if cle:
        logger.info("✓ Clé secrète chargée depuis ANSTAT_SECRET_KEY")
        return cle
    
    # Générer si autorisé
    if generer_si_absent:
        logger.warning("⚠️ Génération d'une clé temporaire pour tests")
        logger.warning("   NE PAS UTILISER EN PRODUCTION !")
        cle = generer_sel_unique()
        
        # Sauvegarder
        with open(fichier_cle, 'w') as f:
            f.write(cle)
        os.chmod(fichier_cle, 0o400)
        logger.info(f"   ✓ Clé sauvegardée dans : {fichier_cle}")
        
        return cle
    
    raise EnvironmentError("Aucune clé secrète trouvée")


def generer_id_anstat(numero_cnps: str, cle_secrete: str) -> str:
    """
    Génère un ID_ANSTAT avec HMAC-SHA256
    
    Args:
        numero_cnps: Numéro CNPS
        cle_secrete: Clé secrète
        
    Returns:
        ID_ANSTAT (64 caractères hex)
    """
    numero = str(numero_cnps).strip()
    key_bytes = cle_secrete.encode("utf-8")
    msg = numero.encode("utf-8")
    mac = hmac.new(key_bytes, msg, digestmod=hashlib.sha256)
    return mac.hexdigest()


def ajouter_id_anstat_polars(
    df: pl.DataFrame,
    cle_secrete: str,
    col_numero: str = "NUMERO_CNPS"
) -> pl.DataFrame:
    """
    Ajoute la colonne ID_ANSTAT avec Polars
    
    Args:
        df: DataFrame avec NUMERO_CNPS
        cle_secrete: Clé secrète
        col_numero: Nom de la colonne NUMERO_CNPS
        
    Returns:
        DataFrame avec ID_ANSTAT
    """
    logger.info("🔐 Génération des ID_ANSTAT...")
    logger.info("   Méthode : HMAC-SHA256")
    
    # Créer une fonction UDF pour Polars
    def hmac_udf(series: pl.Series) -> pl.Series:
        return series.map_elements(
            lambda x: generer_id_anstat(x, cle_secrete),
            return_dtype=pl.Utf8
        )
    
    # Appliquer la fonction
    df_avec_id = df.with_columns(
        hmac_udf(pl.col(col_numero).cast(pl.Utf8)).alias("ID_ANSTAT")
    )
    
    logger.info(f"   ✓ {df_avec_id.height:,} ID_ANSTAT générés")
    
    # Vérifications
    n_uniques = df_avec_id.select(pl.col("ID_ANSTAT").n_unique()).item()
    logger.info(f"\n   Vérifications :")
    logger.info(f"      - ID_ANSTAT uniques : {n_uniques:,}")
    
    if n_uniques == df_avec_id.height:
        logger.info("      ✅ Tous les ID_ANSTAT sont uniques")
    else:
        logger.warning(f"      ⚠️ {df_avec_id.height - n_uniques} collisions détectées")
    
    return df_avec_id


# ============================================================================
# ÉTAPE 4: JOINTURE ET RE-SPLIT
# ============================================================================

def charger_fichier_polars(
    nom_fichier: str,
    dossier: str = DOSSIER_SORTIE
) -> pl.DataFrame:
    """
    Charge un fichier avec Polars (Parquet, CSV ou Excel)
    
    Args:
        nom_fichier: Nom du fichier
        dossier: Dossier contenant le fichier
        
    Returns:
        DataFrame Polars
    """
    chemin = os.path.join(dossier, nom_fichier)
    
    if not Path(chemin).exists():
        # Essayer d'autres extensions
        base = nom_fichier.replace('.xlsx', '').replace('.parquet', '').replace('.csv', '')
        for ext in ['.parquet', '.csv', '.xlsx']:
            chemin_alt = os.path.join(dossier, f"{base}{ext}")
            if Path(chemin_alt).exists():
                chemin = chemin_alt
                break
    
    logger.info(f"📥 Chargement : {chemin}")
    
    ext = Path(chemin).suffix.lower()
    
    if ext == '.parquet':
        df = pl.read_parquet(chemin)
    elif ext == '.csv':
        df = pl.read_csv(chemin)
    elif ext == '.xlsx':
        df = pl.read_excel(chemin)
    else:
        raise ValueError(f"Extension non supportée : {ext}")
    
    logger.info(f"   ✓ {df.height:,} lignes × {df.width} colonnes")
    return df


def joindre_et_resplit(
    df_concat: pl.DataFrame,
    df_ids: pl.DataFrame,
    conserver_numero_cnps: bool = False,
    generer_mensuels: bool = True
) -> pl.DataFrame:
    """
    Joint les ID_ANSTAT et re-split par mois avec Polars
    
    Args:
        df_concat: DataFrame concaténé
        df_ids: DataFrame avec NUMERO_CNPS et ID_ANSTAT
        conserver_numero_cnps: Conserver NUMERO_CNPS après jointure
        generer_mensuels: Générer les fichiers mensuels
        
    Returns:
        DataFrame enrichi
    """
    logger.info("\n🔗 JOINTURE SUR 'NUMERO_CNPS'")
    logger.info("-" * 80)
    
    # Jointure LEFT (très rapide avec Polars)
    df_full = df_concat.join(
        df_ids.select(["NUMERO_CNPS", "ID_ANSTAT"]),
        on="NUMERO_CNPS",
        how="left"
    )
    
    # Statistiques
    nb_total = df_full.height
    nb_sans_id = df_full.filter(pl.col("ID_ANSTAT").is_null()).height
    
    logger.info(f"   ✓ Base enrichie : {nb_total:,} lignes")
    logger.info(f"   → Lignes avec ID_ANSTAT    : {nb_total - nb_sans_id:,}")
    logger.info(f"   → Lignes sans ID_ANSTAT    : {nb_sans_id:,}")
    
    if nb_sans_id == 0:
        logger.info("   ✅ Toutes les lignes ont un ID_ANSTAT")
    else:
        logger.warning(f"   ⚠️ {100 * nb_sans_id / nb_total:.2f}% sans correspondance")
    
    # Supprimer NUMERO_CNPS si demandé
    if not conserver_numero_cnps:
        logger.info("\n🔒 Suppression de NUMERO_CNPS")
        df_full = df_full.drop("NUMERO_CNPS")
        logger.info("   ✅ Colonne supprimée")
    
    # Sauvegarder la base enrichie
    logger.info("\n💾 SAUVEGARDE DE LA BASE ENRICHIE")
    logger.info("-" * 80)
    sauvegarder_polars(df_full, "TRAVAILLEURS_CONCATENES_2024_2025_ID_ANSTAT")
    
    # Re-split par mois
    if generer_mensuels and "PERIODE" in df_full.columns:
        logger.info("\n📆 RE-SPLIT PAR MOIS")
        logger.info("-" * 80)
        
        Path(DOSSIER_MENSUELS_ID).mkdir(parents=True, exist_ok=True)
        
        periodes = df_full.select(pl.col("PERIODE").unique()).sort("PERIODE")
        
        logger.info(f"   Périodes trouvées : {periodes.height}")
        
        for i, periode in enumerate(periodes["PERIODE"], 1):
            logger.info(f"\n   [{i}/{periodes.height}] Période {periode}:")
            
            df_mois = df_full.filter(pl.col("PERIODE") == periode)
            
            # Déterminer le format
            annee, mois = str(periode).split("-")
            nom_fichier = f"TRAVAILLEURS_{annee}_{mois}_ID_ANSTAT"
            
            # Sauvegarder
            fichiers = sauvegarder_polars(df_mois, nom_fichier, DOSSIER_MENSUELS_ID)
            logger.info(f"   ✅ {df_mois.height:,} lignes sauvegardées")
        
        logger.info(f"\n   ✅ Fichiers mensuels dans : {DOSSIER_MENSUELS_ID}/")
    
    return df_full


# ============================================================================
# PIPELINE COMPLET
# ============================================================================

def pipeline_complet(
    nb_fichiers_max: Optional[int] = None,
    conserver_numero_cnps: bool = False,
    generer_mensuels: bool = True
) -> pl.DataFrame:
    """
    Pipeline complet de pseudonymisation avec Polars
    
    Args:
        nb_fichiers_max: Nombre max de fichiers à traiter (None = tous les fichiers)
        conserver_numero_cnps: Conserver NUMERO_CNPS dans la sortie
        generer_mensuels: Générer les fichiers mensuels
        
    Returns:
        DataFrame final avec ID_ANSTAT
    """
    import time
    start_time = time.time()
    
    logger.info("=" * 80)
    logger.info("🚀 PIPELINE COMPLET DE PSEUDONYMISATION (POLARS)")
    logger.info("=" * 80)
    
    try:
        # ÉTAPE 1: Concaténation
        logger.info("\n📥 ÉTAPE 1 : CONCATÉNATION DES FICHIERS MENSUELS")
        logger.info("-" * 80)
        
        fichiers = lister_fichiers_mensuels()
        if not fichiers:
            logger.error("❌ Aucun fichier trouvé")
            return None
        
        df_concat = concatener_fichiers_mensuels(fichiers, nb_fichiers_max)
        
        # Sauvegarder la concaténation
        logger.info("\n💾 Sauvegarde de la base concaténée...")
        sauvegarder_polars(df_concat, "TRAVAILLEURS_CONCATENES_2024_2025")
        
        # ÉTAPE 2: Extraction des uniques
        logger.info("\n🔍 ÉTAPE 2 : EXTRACTION DES NUMERO_CNPS UNIQUES")
        logger.info("-" * 80)
        
        df_uniques = extraire_numero_cnps_uniques(df_concat)
        
        # Sauvegarder les uniques
        logger.info("\n💾 Sauvegarde des NUMERO_CNPS uniques...")
        sauvegarder_polars(df_uniques, "NUMERO_CNPS_UNIQUES")
        
        # ÉTAPE 3: Génération des ID_ANSTAT
        logger.info("\n🔐 ÉTAPE 3 : GÉNÉRATION DES ID_ANSTAT")
        logger.info("-" * 80)
        
        cle_secrete = charger_cle_secrete(generer_si_absent=True)
        df_ids = ajouter_id_anstat_polars(df_uniques, cle_secrete)
        
        # Exemples
        logger.info("\n   Exemples (5 premiers) :")
        for row in df_ids.head(5).iter_rows(named=True):
            logger.info(f"      {row['NUMERO_CNPS']} → {row['ID_ANSTAT']}")
        
        # Sauvegarder les IDs
        logger.info("\n💾 Sauvegarde de la table de mapping...")
        sauvegarder_polars(df_ids, "NUMERO_CNPS_UNIQUES_AVEC_ID_ANSTAT")
        
        # ÉTAPE 4: Jointure et re-split
        logger.info("\n🔗 ÉTAPE 4 : JOINTURE ET RE-SPLIT")
        logger.info("-" * 80)
        
        df_full = joindre_et_resplit(
            df_concat,
            df_ids,
            conserver_numero_cnps,
            generer_mensuels
        )
        
        # Résumé
        elapsed = time.time() - start_time
        logger.info("\n" + "=" * 80)
        logger.info("✅ PIPELINE TERMINÉ AVEC SUCCÈS")
        logger.info(f"⏱️  Temps d'exécution : {elapsed:.2f} secondes ({elapsed/60:.1f} min)")
        logger.info(f"📊 Résultat :")
        logger.info(f"   - Lignes traitées : {df_full.height:,}")
        logger.info(f"   - Colonnes        : {df_full.width}")
        logger.info("=" * 80)
        
        return df_full
        
    except Exception as e:
        logger.error(f"\n❌ ERREUR CRITIQUE: {e}", exc_info=True)
        raise


# ============================================================================
# POINT D'ENTRÉE
# ============================================================================

if __name__ == "__main__":
    # Exemple d'utilisation - Traitement de TOUS les fichiers
    df_final = pipeline_complet(
        nb_fichiers_max=None,           # None = traiter TOUS les fichiers trouvés
        conserver_numero_cnps=False,    # Supprimer NUMERO_CNPS (sécurité)
        generer_mensuels=True           # Générer les fichiers mensuels
    )
    
    logger.info("\n✅ Script terminé. Les fichiers sont disponibles dans ./CNPS_FUSION/")

2025-12-04 20:08:07 - INFO - ================================================================================
2025-12-04 20:08:07 - INFO - 🚀 PIPELINE COMPLET DE PSEUDONYMISATION (POLARS)
2025-12-04 20:08:07 - INFO - ================================================================================
2025-12-04 20:08:07 - INFO - 
📥 ÉTAPE 1 : CONCATÉNATION DES FICHIERS MENSUELS
2025-12-04 20:08:07 - INFO - --------------------------------------------------------------------------------
2025-12-04 20:08:07 - INFO - 📂 Recherche des fichiers dans : C:/Users/f.migone/Documents/12-04-2025-19-31-21_files_list/source
2025-12-04 20:08:07 - INFO - ✓ 3 fichiers mensuels trouvés
2025-12-04 20:08:07 - INFO - Liste des fichiers :
2025-12-04 20:08:07 - INFO -   - TRAVAILLEURS_2024_01_NUMERO_CNPS.xlsx              ( 50.54 MB)
2025-12-04 20:08:07 - INFO -   - TRAVAILLEURS_2024_02_NUMERO_CNPS.xlsx              ( 50.64 MB)
2025-12-04 20:08:07 - INFO -   - TRAVAILLEURS_2024_03_NUMERO_CNPS.xlsx              ( 5